# Model 04: power-system fault classification
Imported from the author-provided Colab notebook on 2026-09-16. Code cells are unchanged; saved outputs and execution metadata were cleared. See `notebooks/README.md` for data setup and `docs/results.md` for separately identified historical evaluations.
Run in Google Colab after configuring your own dataset paths. The external cell currently selects the Indika archive, not the older Laxapana evaluation.


# Cell 01



In [ ]:
# CELL 01 - Imports and Dataset Setup

import zipfile
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models

from google.colab import drive


# Reproducibility
SEED = 42
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)


# Mount Google Drive
drive.mount("/content/drive")


# Dataset paths
DRIVE_ZIP_PATH = Path(
    "/content/drive/MyDrive/FYP_Dataset/Training_2000_14_Bus.zip"
)

EXTRACT_PATH = Path(
    "/content/IEEE14_2000"
)


# Check and extract dataset
if not DRIVE_ZIP_PATH.exists():
    raise FileNotFoundError(f"Dataset ZIP not found: {DRIVE_ZIP_PATH}")

if not EXTRACT_PATH.exists():
    print("Extracting dataset...")

    EXTRACT_PATH.mkdir(parents=True)

    with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)

    print("Extraction completed.")
else:
    print("Using existing extracted dataset.")


# Find all MATLAB files
mat_files = sorted(EXTRACT_PATH.rglob("*.mat"))

if not mat_files:
    raise FileNotFoundError(f"No .mat files found in: {EXTRACT_PATH}")

print("MAT files found:", len(mat_files))
print("Dataset path:", EXTRACT_PATH)

# Cell 02

In [ ]:
# =========================================================
# CELL 02 - Load, add V0/I0, split by simulation, normalize
# =========================================================
import re
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
from sklearn.model_selection import train_test_split

# Configuration
DATASET_NAME, MAT_VARIABLE = "IEEE14_2000", "faultData"
TARGET_LEN, NUM_CHANNELS, NUM_CHANNELS_AUG = 1201, 6, 8
EXPECTED_SAMPLES, SEED = 2000, 42
TEST_SIZE = 0.20
VAL_SIZE = 0.20  # Fraction of remaining groups: ~64/16/20 overall.
STD_FLOOR = 1e-6

SIGNAL_NAMES = ["Va", "Vb", "Vc", "Ia", "Ib", "Ic"]
SIGNAL_NAMES_AUG = SIGNAL_NAMES + ["V0", "I0"]
CLASS_ORDER = [
    "Norm", "AB", "BC", "AC", "AG", "BG", "CG",
    "ABG", "BCG", "ACG", "ABC", "ABCG",
]
LABEL_ALIASES = {label.lower(): label for label in CLASS_ORDER}
LABEL_ALIASES.update({
    "normal": "Norm", "healthy": "Norm", "nofault": "Norm",
    "ba": "AB", "cb": "BC", "ca": "AC",
    "ga": "AG", "gb": "BG", "gc": "CG",
    "bag": "ABG", "cbg": "BCG",
    "cag": "ACG", "gac": "ACG", "gca": "ACG",
})


def label_from_path(path):
    """Prefer the filename; otherwise use the nearest labelled folder."""
    path = Path(path)
    for name in [path.stem] + [p.name for p in path.parents]:
        labels = {
            LABEL_ALIASES[token]
            for token in re.findall(r"[a-z0-9]+", name.lower())
            if token in LABEL_ALIASES
        }
        if len(labels) > 1:
            raise ValueError(f"Ambiguous labels in {path}: {sorted(labels)}")
        if labels:
            return labels.pop()
    raise ValueError(f"No fault label found in {path}")


def simulation_id_from_path(path):
    stem = Path(path).stem.lower()

    case = re.findall(r"(?:^|[_-])case[_-]?(\d+)(?=$|[_-])", stem)
    voltage = re.findall(r"(?<![a-z0-9])(\d+)kv(?![a-z0-9])", stem)

    if len(case) != 1:
        raise ValueError(f"Missing or ambiguous case ID: {path}")
    if len(voltage) > 1:
        raise ValueError(f"Ambiguous voltage ID: {path}")

    voltage_id = f"{int(voltage[0])}kv" if voltage else "no_voltage"
    return f"{voltage_id}_case_{int(case[0])}"


def load_mat_sample(path):
    """Load real signals as (time, 6), preserving precision for phase sums."""
    mat = loadmat(path, variable_names=[MAT_VARIABLE], squeeze_me=True)
    if MAT_VARIABLE not in mat:
        raise ValueError(f"{MAT_VARIABLE!r} missing in {path}")

    sample = np.asarray(mat[MAT_VARIABLE])
    if np.iscomplexobj(sample):
        raise ValueError(f"Expected real time-domain signals: {path}")

    sample = sample.astype(np.float64)
    if sample.shape == (NUM_CHANNELS, TARGET_LEN):
        sample = sample.T
    if sample.shape != (TARGET_LEN, NUM_CHANNELS):
        raise ValueError(f"Invalid shape {sample.shape} in {path}")
    if not np.isfinite(sample).all():
        raise ValueError(f"NaN/Inf in {path}")

    return sample


def add_zero_sequence_channels(signals):
    # Input: Va,Vb,Vc,Ia,Ib,Ic; same units/base within each phase triplet.
    signals = np.asarray(signals, dtype=np.float64)
    v0 = signals[:, :3].mean(axis=1, keepdims=True)
    i0 = signals[:, 3:6].mean(axis=1, keepdims=True)

    sample = np.concatenate((signals, v0, i0), axis=1).astype(np.float32)
    if not np.isfinite(sample).all():
        raise ValueError("Augmented signals exceed the float32 range.")

    return np.ascontiguousarray(sample)


# Load every file; stop on invalid data instead of silently skipping it.
if "mat_files" not in globals():
    raise RuntimeError("Run the dataset extraction cell first.")

paths = sorted(map(Path, mat_files), key=lambda p: str(p))
if not paths:
    raise RuntimeError("No MAT files found.")
if EXPECTED_SAMPLES is not None and len(paths) != EXPECTED_SAMPLES:
    raise RuntimeError(f"Expected {EXPECTED_SAMPLES} files, found {len(paths)}.")

y_text = np.asarray([label_from_path(p) for p in paths])
sim_ids = np.asarray([simulation_id_from_path(p) for p in paths])
file_list = np.asarray([str(p) for p in paths], dtype=object)
X_pu = np.stack([add_zero_sequence_channels(load_mat_sample(p)) for p in paths])
# X_pu keeps the original name; this cell does not convert units to per-unit.


# Exact duplicate audit
hashes = [
    hashlib.blake2b(sample.tobytes(), digest_size=16).hexdigest()
    for sample in X_pu
]
metadata = pd.DataFrame({
    "index": np.arange(len(paths)),
    "file": file_list,
    "sim_id": sim_ids,
    "label": y_text,
    "hash": hashes,
})
duplicates = metadata[metadata["hash"].duplicated(keep=False)]
if not duplicates.empty:
    print(duplicates.sort_values("hash").to_string(index=False))
    raise RuntimeError("Exact duplicate inputs found; resolve them before training.")


# Encode labels
class_names = [label for label in CLASS_ORDER if label in set(y_text)]
label_map = {label: i for i, label in enumerate(class_names)}
y = np.asarray([label_map[label] for label in y_text], dtype=np.int32)
num_classes = len(class_names)


# Stratify simulations, keeping every file from a group together.
grouped = metadata.groupby("sim_id")["label"]
if (grouped.nunique() != 1).any():
    raise ValueError("A simulation ID has multiple labels; check the ID convention.")

simulation_table = grouped.first()
try:
    train_val_groups, test_groups = train_test_split(
        simulation_table.index.to_numpy(),
        test_size=TEST_SIZE,
        stratify=simulation_table.to_numpy(),
        random_state=SEED,
    )
    train_groups, val_groups = train_test_split(
        train_val_groups,
        test_size=VAL_SIZE,
        stratify=simulation_table.loc[train_val_groups].to_numpy(),
        random_state=SEED,
    )
except ValueError as exc:
    raise ValueError(
        "Cannot stratify simulation groups. Check groups per class and split sizes."
    ) from exc

train_ids, val_ids, test_ids = map(set, (train_groups, val_groups, test_groups))
assert not (train_ids & val_ids or train_ids & test_ids or val_ids & test_ids)

train_idx, val_idx, test_idx = [
    np.flatnonzero(np.isin(sim_ids, list(ids)))
    for ids in (train_ids, val_ids, test_ids)
]
assert np.array_equal(
    np.sort(np.concatenate((train_idx, val_idx, test_idx))),
    np.arange(len(paths)),
)
for indices in (train_idx, val_idx, test_idx):
    if set(y[indices]) != set(range(num_classes)):
        raise ValueError("A split is missing a class; increase groups per class.")


# Fit normalization on training samples only. Reuse these in Cell 05.
train_raw = X_pu[train_idx]
channel_mean = train_raw.mean(axis=(0, 1), keepdims=True, dtype=np.float64)
channel_std = train_raw.std(axis=(0, 1), keepdims=True, dtype=np.float64)
channel_std = np.maximum(channel_std, STD_FLOOR)
del train_raw

X = ((X_pu - channel_mean) / channel_std).astype(np.float32)
if not np.isfinite(X).all():
    raise ValueError("Non-finite values after normalization.")

X_train, X_val, X_test = (X[idx] for idx in (train_idx, val_idx, test_idx))
y_train, y_val, y_test = (y[idx] for idx in (train_idx, val_idx, test_idx))
train_files, val_files, test_files = (
    file_list[idx] for idx in (train_idx, val_idx, test_idx)
)

# Compatibility with later cells
CLASS_NAMES = class_names
X_train_s, X_val_s, X_test_s = X_train, X_val, X_test
mean, std = channel_mean, channel_std


# Summary
print(f"{DATASET_NAME}: {X.shape}; channels: {SIGNAL_NAMES_AUG}")
print("Label map:", label_map)

for name, idx in zip(("Train", "Validation", "Test"), (train_idx, val_idx, test_idx)):
    print(
        f"{name}: {len(idx)} samples ({len(idx) / len(X):.1%}), "
        f"{len(np.unique(sim_ids[idx]))} simulations"
    )

print(pd.DataFrame({
    name: np.bincount(y[idx], minlength=num_classes)
    for name, idx in zip(
        ("Train", "Validation", "Test"), (train_idx, val_idx, test_idx)
    )
}, index=class_names).rename_axis("Class"))

print("Checks passed: no exact duplicate inputs; disjoint parsed simulation IDs.")
print("Normalization fitted on training samples only.")

# Training-only diagnostic; no ABC/ABCG separation is assumed.
for label in ("ABC", "ABCG"):
    idx = train_idx[y_text[train_idx] == label]
    if len(idx):
        zero = X_pu[idx, :, 6:8].astype(np.float64)
        rms = np.sqrt(np.mean(zero ** 2, axis=1))
        print(f"{label}: median training RMS [V0, I0] = {np.median(rms, axis=0)}")

# Cell 03

In [ ]:
# =========================================================
# CELL 03 - 3-CNN + BiLSTM, checkpoint and honest diagnostics
# Run after Cell 02. Architecture and training settings retained.
# =========================================================
import json
import time
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from sklearn.metrics import classification_report, confusion_matrix

EPOCHS, BATCH_SIZE, LEARNING_RATE = 30, 128, 5e-4
TRAIN_FS_HZ = 6000.0  # Verified for Training_2000_14_Bus.
RUN_DIR = Path("cnn_bilstm_runs") / (
    datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:6]
)
MODEL_PATH = str(RUN_DIR / "stable_3cnn_bilstm.keras")
PREPROCESS_PATH = str(RUN_DIR / "preprocessing.npz")

for name, features, labels in (
    ("training", X_train_s, y_train), ("validation", X_val_s, y_val)
):
    if features.ndim != 3 or features.shape[1:] != X_train_s.shape[1:]:
        raise ValueError(f"Invalid {name} input shape: {features.shape}")
    if labels.shape != (len(features),) or not len(features):
        raise ValueError(f"Invalid {name} label shape or empty split.")
    if not np.isfinite(features).all():
        raise ValueError(f"Non-finite {name} inputs.")
    if not np.issubdtype(labels.dtype, np.integer):
        raise ValueError("Sparse categorical crossentropy requires integer labels.")
    if labels.min() < 0 or labels.max() >= num_classes:
        raise ValueError(f"Invalid {name} class indices.")

if label_map != {name: i for i, name in enumerate(class_names)}:
    raise ValueError("class_names and label_map disagree.")
if num_classes != len(class_names):
    raise ValueError("num_classes and class_names disagree.")

RUN_DIR.mkdir(parents=True, exist_ok=False)
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
l2 = regularizers.l2(1e-4)

inputs = layers.Input(shape=X_train_s.shape[1:], name="input_signal")
x = inputs
for i, (filters, kernel, dropout) in enumerate(
    ((32, 9, 0.15), (48, 7, 0.20), (64, 5, 0.20)), start=1
):
    x = layers.Conv1D(
        filters, kernel, padding="same", activation="relu",
        kernel_regularizer=l2, name=f"conv{i}"
    )(x)
    x = layers.BatchNormalization(name=f"bn{i}")(x)
    x = layers.MaxPooling1D(2, name=f"pool{i}")(x)
    x = layers.SpatialDropout1D(dropout, name=f"spatial_dropout{i}")(x)

x = layers.Bidirectional(
    layers.LSTM(
        48, return_sequences=True, dropout=0.20,
        recurrent_dropout=0.15, kernel_regularizer=l2
    ), name="bilstm"
)(x)
x = layers.Concatenate()([
    layers.GlobalAveragePooling1D()(x), layers.GlobalMaxPooling1D()(x)
])
x = layers.Dense(64, activation="relu", kernel_regularizer=l2)(x)
x = layers.Dropout(0.40)(x)
outputs = layers.Dense(
    num_classes, activation="softmax", dtype="float32", kernel_regularizer=l2
)(x)
model = models.Model(inputs, outputs, name="Stable_3CNN_BiLSTM")
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE, clipnorm=1.0
    ),
    loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_PATH, monitor="val_loss", mode="min",
        save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", mode="min", patience=10,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", mode="min", factor=0.5,
        patience=4, min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.TerminateOnNaN(),
]

model.summary()
start = time.perf_counter()
history = model.fit(
    X_train_s, y_train, validation_data=(X_val_s, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks,
    shuffle=True, verbose=1
)
training_time = time.perf_counter() - start

val_losses = np.asarray(history.history["val_loss"], dtype=float)
if not len(val_losses) or not np.isfinite(val_losses).all():
    raise RuntimeError("Non-finite validation loss; inspect this run before using it.")
if not Path(MODEL_PATH).is_file():
    raise RuntimeError("This run did not produce a valid checkpoint.")

model = tf.keras.models.load_model(MODEL_PATH)
best_epoch = int(np.argmin(val_losses) + 1)

# Evaluate BOTH splits using the SAME saved model in inference mode.
# Epoch training accuracy has active dropout and changing weights.
train_metrics = model.evaluate(
    X_train_s, y_train, batch_size=BATCH_SIZE, verbose=0, return_dict=True
)
val_metrics = model.evaluate(
    X_val_s, y_val, batch_size=BATCH_SIZE, verbose=0, return_dict=True
)
train_acc, val_acc = train_metrics["accuracy"], val_metrics["accuracy"]
accuracy_gap = train_acc - val_acc
val_predictions = model.predict(
    X_val_s, batch_size=BATCH_SIZE, verbose=0
).argmax(axis=1)

print(f"\nBest epoch by validation loss: {best_epoch}")
print(f"Epoch training accuracy (dropout active): "
      f"{history.history['accuracy'][best_epoch - 1]:.4f}")
print(f"Saved-model training accuracy: {train_acc:.4f}")
print(f"Saved-model validation accuracy: {val_acc:.4f}")
print(f"Inference accuracy gap: {accuracy_gap:.4f}")
print(f"Training time: {training_time / 60:.2f} minutes")
print("These are internal results; they do not establish external generalization.")
print(classification_report(
    y_val, val_predictions, labels=np.arange(num_classes),
    target_names=class_names, digits=4, zero_division=0
))
print("Validation confusion matrix:\n", confusion_matrix(
    y_val, val_predictions, labels=np.arange(num_classes)
))

# Keep preprocessing and class order beside this exact model checkpoint.
np.savez(
    PREPROCESS_PATH,
    channel_mean=np.asarray(channel_mean, dtype=np.float64),
    channel_std=np.asarray(channel_std, dtype=np.float64),
    class_names=np.asarray(class_names, dtype=str),
    signal_names=np.asarray(
        SIGNAL_NAMES_AUG if X_train_s.shape[-1] == 8 else SIGNAL_NAMES, dtype=str
    ),
    target_len=X_train_s.shape[1], raw_channels=6,
    train_fs_hz=TRAIN_FS_HZ, onset_alignment="none",
)
(RUN_DIR / "history.json").write_text(
    json.dumps({k: list(map(float, v)) for k, v in history.history.items()}, indent=2),
    encoding="utf-8"
)
(RUN_DIR / "split_files.json").write_text(json.dumps({
    "train": list(map(str, train_files)),
    "validation": list(map(str, val_files)),
    "test": list(map(str, test_files)),
}, indent=2), encoding="utf-8")
print("Model:", MODEL_PATH)
print("Matching preprocessing:", PREPROCESS_PATH)
# Keep the internal test set out of training/model selection.


In [ ]:
import numpy as np
train_hashes = set(map(tuple, X_train.reshape(len(X_train), -1)[:, ::50].round(4).tolist()))
val_hashes = set(map(tuple, X_val.reshape(len(X_val), -1)[:, ::50].round(4).tolist()))
test_hashes = set(map(tuple, X_test.reshape(len(X_test), -1)[:, ::50].round(4).tolist()))
print('train/val overlap:', len(train_hashes & val_hashes))
print('train/test overlap:', len(train_hashes & test_hashes))
print('X_train len:', len(X_train), 'X_val len:', len(X_val))

In [ ]:
# CELL 04 - Test Accuracy and Confusion Matrix

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    balanced_accuracy_score
)

# Evaluate
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    batch_size=BATCH_SIZE,
    verbose=0
)

# Predictions
y_pred = np.argmax(
    model.predict(X_test, batch_size=BATCH_SIZE, verbose=0),
    axis=1
)

balanced_accuracy = balanced_accuracy_score(y_test, y_pred)

print("Test accuracy:", f"{test_accuracy * 100:.2f}%")
print("Balanced accuracy:", f"{balanced_accuracy * 100:.2f}%")

# Confusion matrix
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(num_classes)
)

fig, ax = plt.subplots(figsize=(9, 7))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
).plot(
    ax=ax,
    xticks_rotation=45,
    values_format="d",
    colorbar=False
)

ax.set_title("Confusion Matrix")
ax.grid(False)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Misclassification summary
# ---------------------------------------------------------
errors = []

for true_id in range(num_classes):
    for predicted_id in range(num_classes):

        count = cm[true_id, predicted_id]

        if true_id != predicted_id and count > 0:
            errors.append({
                "Actual fault": class_names[true_id],
                "Predicted as": class_names[predicted_id],
                "Count": int(count)
            })

errors_df = pd.DataFrame(errors)

print("\nMisclassification Summary")

if errors_df.empty:
    print("No incorrect predictions 🎉")
else:
    errors_df = errors_df.sort_values(
        "Count",
        ascending=False
    ).reset_index(drop=True)

    display(errors_df)


# Cell 05

In [ ]:
# =========================================================
# CELL 05 - Any file count, mixed recording lengths, batched validation
# Works with the existing model + Cell 02, or revised Cell 03.
# =========================================================
import io
import json
import zipfile
from fractions import Fraction
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.signal import resample_poly
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

VALIDATION_SOURCE = Path("/content/drive/MyDrive/FYP_Dataset/Validation(Indika).zip")
# VALIDATION_SOURCE can be a ZIP, a folder (searched recursively), or one MAT file.
# No expected file count: every signal file is accounted for, including the last batch.
INFERENCE_BATCH_SIZE = 128
INVALID_SAMPLE_POLICY = "report"  # Evaluate valid files and report every rejection; "raise" is strict.
TIMEBASE_POLICY = "auto"   # auto: physical timing if available, otherwise relative position.
# "physical" requires timing; "relative" maps every complete recording to model length.
EXTERNAL_FS_HZ = None       # Optional known rate for untimed files; otherwise auto uses relative position.
EXTERNAL_DURATION_S = None  # Alternative: known time from first to last sample.
WINDOW_START_S = 0.0        # Fixed window relative to start of each recording.
TRAIN_FS_HZ = 6000.0        # Verified for the current training dataset.
RAW_SCALE = np.ones(6)      # Known unit/base conversion ONLY; never fit on test data.
RESULTS_DIR = Path("external_validation_results")

if "model" not in globals():
    raise RuntimeError("Load your trained model first.")
if not isinstance(INFERENCE_BATCH_SIZE, int) or INFERENCE_BATCH_SIZE < 1:
    raise ValueError("INFERENCE_BATCH_SIZE must be a positive integer.")
if TIMEBASE_POLICY not in ("auto", "physical", "relative"):
    raise ValueError("TIMEBASE_POLICY must be 'auto', 'physical', or 'relative'.")
if INVALID_SAMPLE_POLICY not in ("report", "raise"):
    raise ValueError("INVALID_SAMPLE_POLICY must be 'report' or 'raise'.")
MODEL_LEN, MODEL_CHANNELS = map(int, model.input_shape[1:])
if MODEL_LEN < 2:
    raise ValueError("Expected at least two time points in the model input.")
if MODEL_CHANNELS not in (6, 8):
    raise ValueError("Expected a model with six or eight input channels.")

# A revised Cell 03 supplies the saved bundle. Existing models use Cell 02 globals.
if "PREPROCESS_PATH" in globals():
    with np.load(PREPROCESS_PATH, allow_pickle=False) as prep:
        norm_mean = prep["channel_mean"]
        norm_std = prep["channel_std"]
        external_class_names = prep["class_names"].tolist()
        TRAIN_FS_HZ = float(prep["train_fs_hz"])
        if int(prep["target_len"]) != MODEL_LEN:
            raise ValueError("Model and preprocessing recording lengths disagree.")
        if str(prep["onset_alignment"]) != "none":
            raise ValueError("This cell requires the matching training alignment pipeline.")
else:
    norm_mean, norm_std = channel_mean, channel_std
    external_class_names = list(class_names)

norm_mean = np.asarray(norm_mean, dtype=np.float64).reshape(1, 1, -1)
norm_std = np.asarray(norm_std, dtype=np.float64).reshape(1, 1, -1)
if norm_mean.shape != (1, 1, MODEL_CHANNELS) or norm_std.shape != norm_mean.shape:
    raise ValueError("Training statistics do not match the model input channels.")
if not (np.isfinite(norm_mean).all() and np.isfinite(norm_std).all()):
    raise ValueError("Non-finite training statistics.")
if np.any(norm_std <= 0):
    raise ValueError("Training standard deviations must be positive.")
if int(model.output_shape[-1]) != len(external_class_names):
    raise ValueError("Model output and saved class order disagree.")
if EXTERNAL_FS_HZ is not None and EXTERNAL_DURATION_S is not None:
    raise ValueError("Declare either external sampling rate or duration, not both.")
if (not np.isfinite(TRAIN_FS_HZ) or TRAIN_FS_HZ <= 0
        or not np.isfinite(WINDOW_START_S) or WINDOW_START_S < 0):
    raise ValueError("Invalid training sample rate or window start.")
RAW_SCALE = np.asarray(RAW_SCALE, dtype=np.float64)
if RAW_SCALE.shape != (6,) or not np.isfinite(RAW_SCALE).all() or np.any(RAW_SCALE == 0):
    raise ValueError("RAW_SCALE must contain six finite, nonzero conversion factors.")

external_label_map = {name: i for i, name in enumerate(external_class_names)}
target_duration = (MODEL_LEN - 1) / TRAIN_FS_HZ


def external_signal_diagnostics(values):
    """Keep source magnitudes in rejection reports; never repair data by clipping."""
    values = np.asarray(values)
    result = {"source_shape": str(values.shape)}
    if np.iscomplexobj(values):
        return result
    try:
        values = values.astype(np.float64)
        finite = np.isfinite(values)
        result["source_nonfinite_values"] = int(np.count_nonzero(~finite))
        result["source_max_abs"] = float(np.max(np.abs(values[finite]))) if finite.any() else np.nan
    except (ValueError, TypeError, OverflowError):
        pass
    return result


def add_external_channels(signals):
    """Keep all channels in float64 until after training-statistic normalization."""
    signals = np.asarray(signals, dtype=np.float64)
    if MODEL_CHANNELS == 6:
        return signals
    # Divide before summing to avoid an overflowing intermediate sum in float64.
    v0 = np.sum(signals[:, :3] / 3.0, axis=1, keepdims=True)
    i0 = np.sum(signals[:, 3:6] / 3.0, axis=1, keepdims=True)
    return np.concatenate((signals, v0, i0), axis=1)


def normalize_external_sample(sample):
    """Check the normalized range BEFORE the one conversion to float32."""
    try:
        with np.errstate(over="raise", invalid="raise", divide="raise"):
            normalized = (sample - norm_mean[0]) / norm_std[0]
    except FloatingPointError as exc:
        raise ValueError(
            f"Normalization overflowed float64; prepared peak={np.max(np.abs(sample)):.6e}. "
            "Inspect signal generation, units, and the saved training statistics."
        ) from exc
    peak = float(np.max(np.abs(normalized)))
    float32_limit = float(np.finfo(np.float32).max)
    if not np.isfinite(normalized).all() or peak > float32_limit:
        raise ValueError(
            f"Normalized peak={peak:.6e}, float32 limit={float32_limit:.6e}. "
            "Inspect the recording and its physical units/bases; no clipping was applied."
        )
    return normalized.astype(np.float32), peak


def iter_external_files(source):
    """Yield (name, byte reader) for every MAT file without extracting the ZIP."""
    source = Path(source)
    if source.is_dir():
        for path in sorted(p for p in source.rglob("*")
                           if p.is_file() and p.suffix.lower() == ".mat"):
            yield path.relative_to(source).as_posix(), path.read_bytes
    elif source.is_file() and source.suffix.lower() == ".zip":
        with zipfile.ZipFile(source) as archive:
            for entry in sorted(archive.infolist(), key=lambda item: item.filename):
                if not entry.is_dir() and entry.filename.lower().endswith(".mat"):
                    yield entry.filename, lambda entry=entry: archive.read(entry)
    elif source.is_file() and source.suffix.lower() == ".mat":
        yield source.name, source.read_bytes
    else:
        raise ValueError(f"Expected an existing ZIP, MAT file, or folder: {source}")


def metadata_sampling_rate(mat):
    """Read common sampling-rate/period fields; reject contradictory declarations."""
    metadata = mat.get("metadata", {})
    records = [metadata, mat] if isinstance(metadata, dict) else [mat]
    rates = []
    for record in records:
        for key in ("fs_Hz", "fs", "Fs", "sampling_rate_hz", "samplingFrequency"):
            if key in record:
                rates.append(float(record[key]))
        for key in ("Ts", "sample_time_s"):
            if key in record:
                period = float(record[key])
                if not np.isfinite(period) or period <= 0:
                    raise ValueError(f"Invalid sampling period: {key}")
                rates.append(1.0 / period)
    if not rates:
        return None
    if not np.isfinite(rates).all() or np.any(np.asarray(rates) <= 0):
        raise ValueError("Invalid source sampling rate.")
    if not np.allclose(rates, rates[0], rtol=1e-3, atol=0):
        raise ValueError("Sampling-rate metadata fields disagree.")
    return rates[0]


def read_timebase(mat, n):
    """Return a physical rate, or None for explicitly identified relative mapping."""
    if TIMEBASE_POLICY == "relative":
        return None, "relative position (timing ignored by policy)"
    metadata_fs = metadata_sampling_rate(mat)
    for key in ("t", "time"):
        if key in mat:
            timestamps = np.asarray(mat[key], dtype=np.float64).squeeze()
            if timestamps.shape != (n,) or not np.isfinite(timestamps).all():
                raise ValueError(f"Invalid {key} timestamp vector.")
            dt = np.diff(timestamps)
            if np.any(dt <= 0):
                raise ValueError("Timestamps must increase strictly.")
            interval = (timestamps[-1] - timestamps[0]) / (n - 1)
            if not np.allclose(dt, interval, rtol=1e-3, atol=1e-9):
                raise ValueError("Irregular timestamps require a separate resampling policy.")
            if metadata_fs is not None:
                if not np.isfinite(metadata_fs) or not np.isclose(
                    metadata_fs, 1.0 / interval, rtol=1e-3, atol=0
                ):
                    raise ValueError("Timestamps and sampling-rate metadata disagree.")
            return 1.0 / interval, f"timestamps:{key}"

    fs = metadata_fs
    source = "sampling metadata"
    if fs is None and EXTERNAL_FS_HZ is not None:
        fs, source = EXTERNAL_FS_HZ, "declared sampling rate"
    if fs is None and EXTERNAL_DURATION_S is not None:
        duration = float(EXTERNAL_DURATION_S)
        if not np.isfinite(duration) or duration <= 0:
            raise ValueError("Declared duration must be positive and finite.")
        fs, source = (n - 1) / duration, "declared duration"
    if fs is None:
        if TIMEBASE_POLICY == "auto":
            return None, "relative position (timing unavailable)"
        raise ValueError("No timebase: set EXTERNAL_FS_HZ or EXTERNAL_DURATION_S, or use TIMEBASE_POLICY='auto'.")
    fs = float(fs)
    if not np.isfinite(fs) or fs <= 0:
        raise ValueError("Invalid source sampling rate.")
    return fs, source


def prepare_external_sample(mat):
    raw = np.asarray(mat[MAT_VARIABLE]).squeeze()
    if raw.ndim != 2 or np.iscomplexobj(raw):
        raise ValueError("Expected a real, two-dimensional signal matrix.")
    if raw.shape[1] != 6 and raw.shape[0] == 6:
        raw = raw.T
    if raw.shape[1] != 6 or raw.shape[0] < 2:
        raise ValueError(f"Expected (time, 6) or (6, time), got {raw.shape}.")
    raw = raw.astype(np.float64)
    if not np.isfinite(raw).all():
        raise ValueError("NaN/Inf in the original recording.")

    n = len(raw)
    fs, time_source = read_timebase(mat, n)
    relative_mapping = fs is None
    if relative_mapping:
        # This is a processing grid, NOT a measured acquisition frequency.
        # Map the first/last source points to the first/last model points.
        fs = (n - 1) / target_duration
    window_start = 0.0 if relative_mapping else WINDOW_START_S
    duration = (n - 1) / fs
    end = window_start + target_duration
    tolerance = 1e-6 / TRAIN_FS_HZ + 1e-7
    if duration + tolerance < end:
        raise ValueError(
            f"Recording spans {duration:.6f}s; the requested window ends at {end:.6f}s. "
            "Missing physical duration cannot be recovered by resizing/padding."
        )

    # Metadata is an audit only; it never chooses a class-dependent crop.
    metadata = mat.get("metadata", {})
    if (not relative_mapping and isinstance(metadata, dict)
            and "FaultStart_s" in metadata and "FaultEnd_s" in metadata):
        origin = 0.0
        for key in ("t", "time"):
            if key in mat:
                origin = float(np.asarray(mat[key]).ravel()[0])
                break
        fault_start = float(metadata["FaultStart_s"]) - origin
        fault_end = float(metadata["FaultEnd_s"]) - origin
        if fault_start > end or fault_end < window_start:
            raise ValueError("The fixed window excludes the recorded fault; review the window policy.")

    # Preserve existing 1201 x 6, 6 kHz recordings exactly.
    start_index = int(round(window_start * fs))
    same_rate = np.isclose(fs, TRAIN_FS_HZ, rtol=1e-6, atol=0)
    aligned_start = np.isclose(start_index / fs, window_start, atol=1e-8, rtol=0)
    if same_rate and aligned_start and start_index + MODEL_LEN <= n:
        raw = raw[start_index:start_index + MODEL_LEN].copy()
        operation = "unchanged" if n == MODEL_LEN and start_index == 0 else "fixed crop"
    else:
        # Low-pass filtering before downsampling avoids aliasing.
        ratio = (Fraction(MODEL_LEN - 1, n - 1) if relative_mapping
                 else Fraction(TRAIN_FS_HZ / fs).limit_denominator(10000))
        actual_fs = fs * ratio.numerator / ratio.denominator
        if not np.isclose(actual_fs, TRAIN_FS_HZ, rtol=1e-6, atol=0):
            raise ValueError("Cannot represent this sampling-rate ratio accurately.")
        converted = resample_poly(
            raw, ratio.numerator, ratio.denominator, axis=0, padtype="line"
        )
        source_t = np.arange(len(converted)) / actual_fs
        target_t = window_start + np.arange(MODEL_LEN) / TRAIN_FS_HZ
        if target_t[-1] > source_t[-1] + tolerance:
            raise ValueError("Resampled recording does not cover the target window.")
        raw = np.column_stack([
            np.interp(target_t, source_t, converted[:, c]) for c in range(6)
        ])
        operation = "resampled" + (" + fixed crop" if duration > end + tolerance else "")

    if relative_mapping:
        operation = "relative length match" if n == MODEL_LEN else "relative-position resample"

    with np.errstate(over="raise", invalid="raise"):
        raw *= RAW_SCALE
        sample = add_external_channels(raw)
    if sample.shape != (MODEL_LEN, MODEL_CHANNELS) or not np.isfinite(sample).all():
        raise ValueError("Invalid prepared model input.")
    return sample, {
        "original_samples": n,
        "source_fs_hz": np.nan if relative_mapping else fs,
        "duration_s": np.nan if relative_mapping else duration,
        "time_source": time_source, "operation": operation,
        "relative_mapping": relative_mapping,
        "prepared_max_abs": float(np.max(np.abs(sample))),
    }


def predict_external_batch(samples):
    """Samples are already normalized; isolate invalid model outputs by record."""
    normalized = np.stack(samples)
    scores = np.asarray(model.predict(
        normalized, batch_size=INFERENCE_BATCH_SIZE, verbose=0
    ))
    if scores.shape != (len(samples), len(external_class_names)):
        raise ValueError("Model output shape does not match the saved class order.")
    valid = np.isfinite(scores).all(axis=1)
    valid &= ((scores >= 0) & (scores <= 1)).all(axis=1)
    with np.errstate(over="ignore", invalid="ignore"):
        valid &= np.isclose(scores.sum(axis=1), 1.0, rtol=1e-4, atol=1e-6)
    squares = np.sum(normalized[valid].astype(np.float64) ** 2, axis=(0, 1))
    return scores, valid, squares


# Record count is discovered from the input; waveform memory is bounded by batch size.
external_labels, rows, failures, auxiliary_files = [], [], [], []
pending_samples, pending_rows, probability_batches = [], [], []
normalized_squares = np.zeros(MODEL_CHANNELS, dtype=np.float64)


def flush_external_batch():
    global normalized_squares
    if not pending_samples:
        return
    scores, valid, squares = predict_external_batch(pending_samples)
    normalized_squares += squares
    if valid.any():
        probability_batches.append(scores[valid])
    for record, accepted in zip(pending_rows, valid):
        if accepted:
            rows.append(record)
            external_labels.append(record["true_label"])
        else:
            failures.append({
                **record, "stage": "prediction",
                "reason": "Model produced non-finite or invalid probabilities; inspect input scale and model numerics.",
            })
    pending_samples.clear()
    pending_rows.clear()


mat_file_count = 0
for name, read_bytes in iter_external_files(VALIDATION_SOURCE):
    mat_file_count += 1
    stage, label, diagnostic, info = "loading", None, {}, {}
    try:
        label = label_from_path(name)
    except ValueError:
        pass
    try:
        mat = loadmat(
            io.BytesIO(read_bytes()), simplify_cells=True,
            variable_names=[MAT_VARIABLE, "t", "time", "metadata", "fs_Hz", "fs", "Fs",
                            "sampling_rate_hz", "samplingFrequency", "Ts", "sample_time_s"]
        )
        if MAT_VARIABLE not in mat:
            # Only skip non-sample metadata files with no recognized fault label.
            try:
                label_from_path(name)
            except ValueError:
                auxiliary_files.append(name)
                continue
            raise ValueError(f"Labelled sample is missing {MAT_VARIABLE}.")
        diagnostic = external_signal_diagnostics(mat[MAT_VARIABLE])
        stage = "label"
        label = label_from_path(name)
        if label not in external_label_map:
            raise ValueError(f"Untrained class: {label}")
        stage = "preprocessing"
        with np.errstate(over="raise", invalid="raise", divide="raise"):
            sample, info = prepare_external_sample(mat)
        stage = "normalization"
        normalized, peak = normalize_external_sample(sample)
        pending_samples.append(normalized)
        pending_rows.append({
            "file": name, "true_label": label, **diagnostic, **info,
            "normalized_max_abs": peak,
        })
    except (ValueError, OSError, TypeError, NotImplementedError, OverflowError,
            FloatingPointError, zipfile.BadZipFile) as exc:
        failures.append({"file": name, "true_label": label, "stage": stage,
                         "reason": str(exc), **diagnostic, **info})
        continue
    if len(pending_samples) == INFERENCE_BATCH_SIZE:
        flush_external_batch()

# Include the final partial batch, even when the dataset contains just one file.
flush_external_batch()

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
failure_columns = ["file", "true_label", "stage", "reason", "source_shape",
                   "source_nonfinite_values", "source_max_abs", "prepared_max_abs", "normalized_max_abs"]
failure_report = pd.DataFrame(failures).reindex(columns=failure_columns)
failure_report.to_csv(RESULTS_DIR / "rejected_files.csv", index=False)
# Clear results from an earlier run even if this run cannot report an accuracy.
pd.DataFrame(columns=["file", "true_label", "predicted_label", "confidence", "correct"]).to_csv(
    RESULTS_DIR / "external_predictions.csv", index=False
)
attempted_count = mat_file_count - len(auxiliary_files)
if len(rows) + len(failures) != attempted_count:
    raise RuntimeError("Some source files were not accounted for in the evaluation audit.")
coverage = len(rows) / attempted_count if attempted_count else 0.0
summary = {
    "source": str(VALIDATION_SOURCE), "mat_files_found": mat_file_count,
    "status": "failed" if not rows or (failures and INVALID_SAMPLE_POLICY == "raise") else
              ("partial" if failures else "complete"),
    "invalid_sample_policy": INVALID_SAMPLE_POLICY, "timebase_policy": TIMEBASE_POLICY,
    "auxiliary_files": len(auxiliary_files), "attempted_recordings": attempted_count,
    "evaluated_recordings": len(rows), "rejected_recordings": len(failures),
    "coverage": coverage, "accuracy_evaluated": None,
    "correct_fraction_all_attempted": None,
}
(RESULTS_DIR / "evaluation_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
if failures:
    print(f"Rejected {len(failures)}/{attempted_count} recordings. Details: "
          f"{RESULTS_DIR / 'rejected_files.csv'}")
    print(failure_report[["file", "stage", "reason"]].head(10).to_string(index=False))
    if INVALID_SAMPLE_POLICY == "raise":
        raise RuntimeError("Strict mode: rejected recordings found. Reports saved; no accuracy reported.")
if not rows:
    raise RuntimeError("No evaluable recordings. Inspect rejected_files.csv; no accuracy can be calculated.")

probabilities = np.concatenate(probability_batches, axis=0)
y_external_text = np.asarray(external_labels)
y_external = np.asarray([external_label_map[label] for label in external_labels], dtype=np.int32)
if len(probabilities) != len(y_external):
    raise RuntimeError("Prediction count differs from the number of loaded recordings.")

audit = pd.DataFrame(rows)
file_names = audit["file"].to_numpy()
print(f"MAT files found: {mat_file_count}; evaluated recordings: {len(audit)}; "
      f"auxiliary MAT files: {len(auxiliary_files)}")
print(f"Evaluation coverage: {len(audit)}/{attempted_count} ({coverage:.2%}); "
      f"rejected: {len(failures)}")
status_rows = [{"Class": record["true_label"], "Status": "Evaluated"} for record in rows]
status_rows += [{"Class": record.get("true_label") or "Unknown", "Status": "Rejected"}
               for record in failures]
status_table = pd.DataFrame(status_rows)
print(pd.crosstab(status_table["Class"], status_table["Status"]).reindex(
    columns=["Evaluated", "Rejected"], fill_value=0
))
print(f"Model input per recording: ({MODEL_LEN}, {MODEL_CHANNELS}); "
      f"training reference: {target_duration:.3f}s at {TRAIN_FS_HZ:g} Hz")
print(audit.groupby(["original_samples", "operation"]).size().rename("files"))
relative_count = int(audit["relative_mapping"].sum())
if relative_count:
    print(f"Timing fallback: {relative_count}/{len(audit)} recordings mapped by relative position. "
          "Their complete signals are fitted to the model length; their physical timing "
          "is unavailable or ignored by policy. Scores below apply to this conversion.")
print("External normalized per-channel RMS:",
      np.sqrt(normalized_squares / (len(audit) * MODEL_LEN)))

# Names are a useful warning; equal names alone do not prove duplicate signals.
if "file_list" in globals():
    internal_names = {Path(str(p)).name for p in file_list}
    name_overlap = sum(Path(name).name in internal_names for name in audit["file"])
    if name_overlap:
        print(f"Audit: {name_overlap} filenames also occur internally; check scenario provenance.")

y_pred = probabilities.argmax(axis=1)
accuracy = accuracy_score(y_external, y_pred)
cm = confusion_matrix(y_external, y_pred, labels=np.arange(len(external_class_names)))
class_support = cm.sum(axis=1)
present_classes = class_support > 0
# Macro recall over observed classes also works for a one-record dataset.
balanced_accuracy = float(np.mean(np.diag(cm)[present_classes] / class_support[present_classes]))
macro_f1 = f1_score(
    y_external, y_pred, labels=np.unique(y_external), average="macro", zero_division=0
)
score_context = " (includes relative-position mapping)" if relative_count else ""
accuracy_including_failures = int(np.count_nonzero(y_pred == y_external)) / attempted_count
print(f"\nAccuracy on evaluated recordings{score_context}: {accuracy:.2%}")
if failures:
    print(f"Correct / all attempted recordings: {accuracy_including_failures:.2%} "
          "(rejected recordings counted as unsuccessful)")
print(f"Balanced accuracy (classes present): {balanced_accuracy:.2%}")
print(f"Macro F1 (classes present): {macro_f1:.4f}")
print(classification_report(
    y_external, y_pred, labels=np.arange(len(external_class_names)),
    target_names=external_class_names, digits=4, zero_division=0
))

audit["predicted_label"] = np.asarray(external_class_names)[y_pred]
audit["confidence"] = probabilities.max(axis=1)
audit["correct"] = y_pred == y_external
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
audit.to_csv(RESULTS_DIR / "external_predictions.csv", index=False)
summary.update({
    "accuracy_evaluated": float(accuracy), "balanced_accuracy_evaluated": balanced_accuracy,
    "macro_f1_evaluated": float(macro_f1),
    "correct_fraction_all_attempted": accuracy_including_failures,
    "relative_position_evaluated": relative_count,
})
(RESULTS_DIR / "evaluation_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(cm, display_labels=external_class_names).plot(
    ax=ax, xticks_rotation=45, colorbar=False
)
ax.set_title(f"External validation — {len(audit)}/{attempted_count} recordings evaluated")
ax.grid(False)
plt.tight_layout()
plt.show()
